# E25 — small development test (not Table 17)
Use 1,024 factual training episodes (four shards, reuse the completed official collection shards), 1,000 optimizer steps per model, 16 x 16 population parents/draws and 20 episodes per condition. All six arms share the same model and paired test episodes. Separate development seeds and output directory; no official test exposure or SOTA claim. No need to finish all 8,192 episodes. Stop the old collection without resetting the runtime before running this notebook. Existing GPU Python environment and encoder cache reused.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
from importlib.metadata import version

CACHE_ROOT = Path('/kaggle/working' if Path('/kaggle/working').exists() else '/content') / 'e25_gpu_v3'
ROOT = CACHE_ROOT / 'dev_1024_v1'
TRAIN_N = 1024
FIT_STEPS = 1000
TEST_N = 20
ROOT.mkdir(parents=True, exist_ok=True)
UP = CACHE_ROOT / 'counterfactual-WM'
BUNDLE = ROOT / 'e25'
BUNDLE.mkdir(exist_ok=True)
PY = sys.executable
PIN = '5c84bba92b2ff19d41175070a61238fa1396d10d'
E25_PIN = 'e6c3dff96bd9a0592ba91e908c757282f3c74a4b'
ENV = dict(os.environ, CWM_OUTPUTS=str(ROOT/'outputs'), MUJOCO_GL='egl',
           PYOPENGL_PLATFORM='egl', SDL_VIDEODRIVER='dummy', WANDB_MODE='disabled',
           PYTHONUNBUFFERED='1', HF_HOME=str(CACHE_ROOT/'hf_cache'))
def shell(*args, cwd=ROOT, capture=False):
    args = list(map(str, args))
    print('>', ' '.join(args), flush=True)
    return subprocess.run(args, cwd=cwd, env=ENV, check=True, text=True,
                          stdout=subprocess.PIPE if capture else None).stdout

def py(*args, capture=False):
    return shell(PY, *args, cwd=UP, capture=capture)

shell('nvidia-smi')
shell(PY, '-c', "import torch; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available(), 'CPU-only torch or no GPU: select GPU and use its CUDA-enabled Python environment'; print(torch.cuda.get_device_name(0)); print(torch.ones(4,device='cuda').sum().item())")
print('Free disk GiB:', round(shutil.disk_usage(ROOT).free/2**30, 1))

In [ ]:
# Download only the selected source revision and install dependencies for HandReach.
if not UP.exists():
    shell('git', 'clone', 'https://github.com/galilai-group/counterfactual-WM', UP)
shell('git', 'checkout', '--detach', PIN, cwd=UP)
assert shell('git', 'rev-parse', 'HEAD', cwd=UP, capture=True).strip() == PIN
constraints = ROOT/'constraints.txt'
constraints.write_text('\n'.join(f'{p}=={version(p)}' for p in ('torch','torchvision','numpy'))+'\n')
# Do not reinstall already-matching dependencies in the existing runtime.
from importlib.metadata import distribution, PackageNotFoundError
required={'mujoco':'3.11.0','gymnasium':'1.2.3','gymnasium-robotics':'1.4.2',
          'timm':'1.0.26','transformers':'5.4.0'}
try:
    origin=json.loads(distribution('stable-pretraining').read_text('direct_url.json') or '{}')
    matched=all(version(k)==v for k,v in required.items()) and origin.get('vcs_info',{}).get('commit_id')=='ab836bf699a2be712dfcf980a5eb70d36391e876'
except (PackageNotFoundError,ValueError):
    matched=False
if not matched:
    py('-m','pip','install','-q','-c',constraints,
       'mujoco==3.11.0','gymnasium==1.2.3','gymnasium-robotics==1.4.2',
       'timm==1.0.26','transformers==5.4.0',
       'stable-pretraining @ git+https://github.com/galilai-group/stable-pretraining@ab836bf699a2be712dfcf980a5eb70d36391e876')
    py('-m','pip','install','-q','--no-deps','-e',UP)
else:
    print('Reusing installed HandReach dependencies')
    py('-m','pip','install','-q','--no-deps','-e',UP)
py('-c', "import torch; from cwm.model import WorldModel; from cwm import train; assert torch.cuda.is_available(); print('Imports and GPU OK')")

In [ ]:
# Fetch the committed E25 sources. Cache the posterior solve once per observation.
import urllib.request
for name in ('e25.py','run_e25.py','score_e25.py','test_e25.py'):
    url = f'https://raw.githubusercontent.com/shekhawatxyz/fastai-2022/{E25_PIN}/experiments/recursive_world/e25/{name}'
    with urllib.request.urlopen(url, timeout=60) as r:
        (BUNDLE/name).write_bytes(r.read())
p = BUNDLE/'e25.py'
s = p.read_text()
def replace_once(s, old, new):
    assert s.count(old) == 1, f'Unexpected source revision: {old}'
    return s.replace(old, new, 1)
s = replace_once(s, 'self.raw_last = torch.zeros_like(self.sigma)',
                 'self.raw_last = torch.zeros_like(self.sigma)\n        self._cached = None')
s = replace_once(s, 'action = action.reshape(-1)',
                 'self._cached = None\n        action = action.reshape(-1)')
s = replace_once(s, 'p = self.sigma.new_zeros(2*self.a+self.d)\n        p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n        return p',
                 'if self._cached is None:\n            p = self.sigma.new_zeros(2*self.a+self.d)\n            p[self.mask] = torch.linalg.solve(self.prec, self.rhs)\n            self._cached = p\n        return self._cached')
p.write_text(s)
# Resume complete records rather than restarting the whole expensive evaluation.
p = BUNDLE/'run_e25.py'; s = p.read_text()
s = replace_once(s, "with args.out.open('x') as f: # no overwrite or accidental duplicate episodes",
"""expected_keys = {(i,c,a) for i in range(100*args.seed,100*args.seed+args.count)
                         for c in design['conditions'] for a in arms}
        done = set()
        if args.out.exists():
            raw = args.out.read_bytes()
            complete = raw[:raw.rfind(b'\n')+1]
            for line in complete.splitlines():
                r = json.loads(line)
                key = (r['episode'],r['condition'],r['arm'])
                if (key not in expected_keys or key in done or r['mode'] != mode
                    or r['seed_base'] != design['seed_base']
                    or any(r.get(k) != v for k,v in tag.items())
                    or r['noise'] != list(map(float,noise)) or r['alpha'] != alpha):
                    raise ValueError('Incompatible previous output: use a fresh directory')
                done.add(key)
            if complete != raw: args.out.write_bytes(complete)
        with args.out.open('a') as f:""".replace("b'\n'", "b'\\n'"))
s = replace_once(s, "r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)",
                 "if (episode,condition,arm) in done: continue\n                        r=run_episode(up,sim,m,design,d['coordinates'],sigma,episode,condition,arm,noise,alpha)")
s = replace_once(s, "f.write(json.dumps(r)+'\\n');f.flush()", "f.write(json.dumps(r)+'\\n');f.flush()\n                print(mode, episode+1, '/ 100', flush=True)")
compile(s,str(p),'exec'); p.write_text(s)
py(BUNDLE/'test_e25.py')
# Move development episodes away from the official test seed base.
p = BUNDLE/'run_e25.py'
s = p.read_text()
s = replace_once(s, "original_base=design['seed_base'];", "design['seed_base']+=3_000_000;original_base=design['seed_base'];")
s = replace_once(s, "print(mode, episode+1, '/ 100', flush=True)", "print(mode, episode+1, '/', args.count, flush=True)")
compile(s,str(p),'exec'); p.write_text(s)


In [ ]:
# Fail early if MuJoCo EGL or the pinned encoder cannot run.
py('-c', """
from cwm import registry, encoders
from cwm.sims import make
import numpy as np, torch
sim=make(registry.simulator('hand','population'))
try:
    sim.reset(2201000000)
    frame=sim.render()
    print('Rendered', frame.shape)
    assert frame.shape == (224,224,3)
finally: sim.close()
model=encoders.load('mae_cls','cuda')
z=encoders.encode(model,np.asarray([frame]),batch_size=1)
assert torch.isfinite(z).all()
print('Pinned encoder OK',z.shape)
""")
# Capture the actual installed versions for reproducibility.
(ROOT/'environment.txt').write_text(py('-m','pip','freeze',capture=True))

In [ ]:
# Reuse existing completed shards; encode them directly without merging raw images.
shared = CACHE_ROOT/'outputs/data/hand'
if not (shared/'rollouts.npz').exists():
    for start in range(0, TRAIN_N, 256):
        shard = shared/'shards'/f'{start:06d}'
        if not (shard/'rollouts.npz').is_file():
            print('Collecting missing development shard', start, flush=True)
            subprocess.run([PY,'experiments/2_data/collect.py','hand','--index',str(start//256)],
                           cwd=UP,env=dict(ENV,CWM_OUTPUTS=str(CACHE_ROOT/'outputs')),check=True)

py('-c', f"""
from pathlib import Path
import numpy as np
import torch
from cwm import registry, encoders, io
shared=Path({str(CACHE_ROOT/'outputs/data/hand')!r})
n={1024}
out=registry.data('hand')
out.mkdir(parents=True,exist_ok=True)
feature=registry.features('hand','mae_cls')
if not (out/'rollouts.npz').exists() or not feature.exists():
    if (shared/'rollouts.npz').is_file():
        parts=[(shared,n)]
    else:
        parts=[(shared/'shards'/f'{{i:06d}}',256) for i in range(0,n,256)]
    model=encoders.load('mae_cls','cuda')
    zs=[]; rows=[]
    for part,k in parts:
        rgb=np.load(part/'frames.npy',mmap_mode='r')[:k]
        assert rgb.shape==(k,4,224,224,3),rgb.shape
        with np.load(part/'rollouts.npz') as r:
            rows.append({{key:r[key][:k] if key!='coordinates' else r[key] for key in r.files}})
        z=encoders.encode(model,rgb.reshape(-1,224,224,3),batch_size=32).reshape(k,4,-1)
        zs.append(z)
        print('Encoded episodes',sum(len(x) for x in zs),'/',n,flush=True)
    assert all(np.array_equal(r['coordinates'],rows[0]['coordinates']) for r in rows)
    io.savez(out/'rollouts.npz',coordinates=rows[0]['coordinates'],
             **{{key:np.concatenate([r[key] for r in rows]) for key in rows[0] if key!='coordinates'}})
    io.save(dict(z=torch.cat(zs),encoder='mae_cls',state_sha256=model.spec['state_sha256']),feature)
assert len(io.load(feature)['z'])==n
""")
# Train the original architectures with a deliberately short development budget.
train_code = """
import numpy as np
from cwm import registry, io, readout, predictor
import sys
steps=int(sys.argv[2])
d=registry.dataset('hand')
z=io.load(registry.features('hand','mae_cls'))['z']
with np.load(registry.data('hand','rollouts.npz')) as r:
    actions=r['actions']; physical=r['physical'][...,d['coordinates']]; epsilon=r['epsilon']
if sys.argv[1]=='readout':
    out=registry.readout('hand','mae_cls','state').parent
    if not (out/'readout.pt').exists():
        readout.fit(out,'state',z,actions,physical,epsilon,anchor_time=2,
                    hidden=[1024,1024,1024,2],steps=steps,validation_interval=100,device='cuda')
else:
    out=registry.OUT/'fits/hand_mae_cls_h1_s0'
    if not (out/'fit.pt').exists():
        predictor.fit(out,z,actions,history=1,anchor_time=2,anchor_only=False,
                      hidden=[1024,1024],weight_decay=.3,seed=0,steps=steps,
                      validation_interval=100,device='cuda')
"""
from concurrent.futures import ThreadPoolExecutor
gpu_count=int(py('-c','import torch; print(torch.cuda.device_count())',capture=True).strip())
def train_one(role,gpu):
    subprocess.run([PY,'-c',train_code,role,str(FIT_STEPS)],cwd=UP,
                   env=dict(ENV,CUDA_VISIBLE_DEVICES=str(gpu)),check=True)
visible=ENV.get('CUDA_VISIBLE_DEVICES')
# Preserve runtime device mapping (when set), rather than assuming physical IDs.
gpu_ids=visible.split(',') if visible and visible!='-1' else [str(i) for i in range(gpu_count)]
if gpu_count>=2:
    with ThreadPoolExecutor(2) as pool:
        jobs=[pool.submit(train_one,role,gpu_ids[i]) for i,role in enumerate(('readout','predictor'))]
        for job in jobs: job.result()
else:
    for role in ('readout','predictor'): train_one(role,gpu_ids[0])


In [ ]:
# Small, independent population for distance normalization, no image file.
py('-c', """
from cwm import registry
from cwm.sims import population
out=registry.data('hand','population')
if not (out/'results.json').exists():
    import shutil
    shutil.rmtree(out,ignore_errors=True)
    spec=registry.simulator('hand','population')
    spec.update(parents=16,fields=16,save_frames=False,seed=spec['seed']+3_000_000)
    population.run(spec,out)
assert (out/'branches.npz').exists()
print('Development population complete')
""")


In [ ]:
# Compare E25's two unchanged arms with the upstream Planner on disjoint smoke seeds.
# Only implementation parity is checked here, not E25's research hypothesis.
py('-c', f"""
import sys, importlib.util, numpy as np
from pathlib import Path
sys.path.insert(0,{str(BUNDLE)!r})
from cwm import registry, io
from cwm.model import WorldModel
from cwm.sims.worker import Remote
from run_e25 import run_episode
spec=importlib.util.spec_from_file_location('planning', 'experiments/4_evaluate/planning.py')
up=importlib.util.module_from_spec(spec); spec.loader.exec_module(up)
d=io.read_json(registry.CONFIGS/'planning/replan.json'); d['seed_base']+=5000000
fit=registry.OUT/'fits/hand_mae_cls_h1_s0/fit.pt'
m=WorldModel('mae_cls',fit,registry.readout('hand','mae_cls','state'),'cuda')
sigma=up.sigma_of('hand'); idx=registry.dataset('hand')['coordinates']
sim=Remote(registry.simulator('hand','population'),up.sim_command('hand'))
try:
    planner=up.Planner(sim,m,d,idx,sigma)
    for condition in d['conditions']:
        for arm in d['arms']:
            goal=sim('fix',np.random.default_rng(np.random.SeedSequence([d['seed_base'],0,14])).uniform(-planner.limit,planner.limit,m.a))
            ref=planner.replan(0,condition,arm,goal)
            row=run_episode(up,sim,m,d,idx,sigma,0,condition,arm,np.ones_like(sigma),0.)
            np.testing.assert_allclose(row['distances'],ref['distances'],atol=1e-5,rtol=0)
    print('Six smoke baseline parity checks passed')
finally: sim.close()
""")

In [ ]:
# Paired development comparison. Do NOT use the official scorer or seeds.
cal=ROOT/'calibration/hand_mae_cls_s0.jsonl'
test=ROOT/'test/hand_mae_cls_s0.jsonl'
base=[BUNDLE/'run_e25.py','--upstream',UP,'--dataset','hand','--encoder','mae_cls','--seed',0,'--device','cuda','--count',TEST_N]
py(*base,'--out',cal)
py(*base,'--calibration',cal,'--out',test)
import numpy as np
rows=[json.loads(line) for line in test.read_text().splitlines()]
arms=('trained_predictor','counterfactual','scalar','bias','actuator','joint')
conditions=('persistent','fresh','none')
keys={(r['episode'],r['condition'],r['arm']):r for r in rows}
assert len(rows)==len(keys)==TEST_N*len(arms)*len(conditions)
assert all(r['mode']=='test' and r['seed_base']==3112000000 for r in rows)
assert set(keys)=={(i,c,a) for i in range(TEST_N) for c in conditions for a in arms}
success=np.array([[[keys[i,c,a]['success'] for a in arms] for c in conditions] for i in range(TEST_N)],dtype=float)
dist=np.array([[[keys[i,c,a]['final_distance'] for a in arms] for c in conditions] for i in range(TEST_N)])
assert np.isfinite(dist).all()
assert np.array_equal(success,dist<.2)
delta=success[:,:,5]-success[:,:,1]
rng=np.random.default_rng(250026)
boot=delta[rng.integers(TEST_N,size=(2000,TEST_N))].mean(axis=(1,2))
report=dict(scope='development_only',training_episodes=TRAIN_N,fit_steps=FIT_STEPS,
            test_episodes_per_condition=TEST_N,records=len(rows),
            success_pct={c:dict(zip(arms,(100*success[:,j,:].mean(0)).tolist())) for j,c in enumerate(conditions)},
            mean_distance={c:dict(zip(arms,dist[:,j,:].mean(0).tolist())) for j,c in enumerate(conditions)},
            joint_minus_counterfactual_pp=float(100*delta.mean()),
            descriptive_paired_ci_pp=(100*np.quantile(boot,[.025,.975])).tolist(),
            note='Reduced fits and distance scale; smoke parity only; not an official or SOTA result.')
(ROOT/'score.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


In [ ]:
# Export the evidence (also works after partial execution).
import zipfile
archive=ROOT/'e25_dev_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for folder in (BUNDLE,ROOT/'calibration',ROOT/'test',ROOT/'outputs/planning/replan'):
        if folder.exists():
            for p in folder.rglob('*'):
                if p.is_file() and p.suffix in ('.py','.jsonl','.json'):
                    z.write(p,p.relative_to(ROOT))
    for p in (ROOT/'environment.txt',ROOT/'score.json'):
        if p.exists(): z.write(p,p.name)
from IPython.display import FileLink, display
print(archive)
display(FileLink(str(archive)))
# Portable development cache for later experiments; no raw RGB or encoder download.
cache=ROOT/'e25_dev_cache.zip'
with zipfile.ZipFile(cache,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for relative in ('data/hand/rollouts.npz','data/hand/features/mae_cls.pt',
                     'data/hand/population/branches.npz','data/hand/population/results.json',
                     'data/hand/population/resolved.json','fits/hand_mae_cls_h1_s0/fit.pt',
                     'fits/hand_mae_cls_h1_s0/fit.json','readouts/hand/mae_cls/state/readout.pt',
                     'readouts/hand/mae_cls/state/results.json'):
        p=ROOT/'outputs'/relative
        if p.is_file(): z.write(p,'outputs/'+relative)
    if (ROOT/'environment.txt').exists(): z.write(ROOT/'environment.txt','environment.txt')
    z.writestr('manifest.json',json.dumps(dict(upstream=PIN,e25=E25_PIN,train_n=TRAIN_N,fit_steps=FIT_STEPS,scope='development_only')))
print('Cache:',cache,'size MiB:',round(cache.stat().st_size/2**20,1))
display(FileLink(str(cache)))
